Reading Files

In [1]:
import pandas as pd
all_Sheets = pd.read_excel('Under Develop.xlsx', sheet_name=None)

In [2]:
all_Sheets.keys()

dict_keys(['railway', 'Train_Transactions', 'Journeys', 'Stations', 'Route_Stations', 'Routes'])

Sheets Into Dataframes

In [3]:
df_railway = all_Sheets['railway']
df_transactions = all_Sheets['Train_Transactions']
df_journeys = all_Sheets['Journeys']
df_stations = all_Sheets['Stations']
df_route_stations = all_Sheets['Route_Stations']
df_routes = all_Sheets['Routes']

In [5]:
df_journeys

,Journey_ID,Journey_status,departure_time,arrival_time,actual_arrival_time,time_period,reason_for_delay,delay_minutes,scheduled_duration,actual_duration,departure_station,arrival_destination,day_of_departure,date_of_journey,Route_ID
0,1,On Time,18:45:00,19:05:00,19:05:00,Evening,No Delay,0.0,20,20.0,BHM,COV,Sunday,2024-01-07,43
1,2,On Time,18:45:00,19:05:00,19:05:00,Evening,No Delay,0.0,20,20.0,BHM,COV,Tuesday,2024-01-16,43
2,3,On Time,18:45:00,19:05:00,19:05:00,Evening,No Delay,0.0,20,20.0,BHM,COV,Friday,2024-02-09,43
3,4,Cancelled,18:45:00,19:05:00,NaN,Evening,Staffing,NaN,20,NaN,BHM,COV,Wednesday,2024-02-21,43
4,5,On Time,18:45:00,19:05:00,19:05:00,Evening,No Delay,0.0,20,20.0,BHM,COV,Thursday,2024-04-04,43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19866,19867,On Time,16:00:00,17:10:00,17:10:00,Afternoon,No Delay,0.0,70,70.0,YRK,PBO,Saturday,2024-03-23,22
19867,19868,On Time,18:45:00,19:55:00,19:55:00,Evening,No Delay,0.0,70,70.0,YRK,PBO,Saturday,2024-03-30,22
19868,19869,Delayed,07:30:00,07:55:00,08:13:00,Morning,Staffing,18.0,25,43.0,YRK,WKF,Thursday,2024-01-25,41
19869,19870,Delayed,07:30:00,07:55:00,08:00:00,Morning,Staffing,5.0,25,30.0,YRK,WKF,Wednesday,2024-04-03,41


In [6]:
mask = (df_journeys['delay_minutes'] == 0) & (df_journeys['Journey_status'] != 'On Time')
df_journeys.loc[mask, 'Journey_status'] = 'On Time'
df_journeys.loc[mask, 'reason_for_delay'] = 'No Delay'


In [7]:
df_journeys['Journey_status'].value_counts()

Journey_status
On Time      18033
Delayed       1048
Cancelled      790
Name: count, dtype: int64

--1 Total Tickets Sold

In [5]:
total_tickets = df_transactions['Transaction_ID'].nunique()
total_tickets

31653

--2 Average Ticket Price

In [6]:
avg_price = df_transactions['Price'].mean()
avg_price

np.float64(23.439200075822196)

--3 Purchase Type Distribution


In [7]:
df_transactions['Purchase_Type'].value_counts()


Purchase_Type
Online     18521
Station    13132
Name: count, dtype: int64

--4 Payment Method Distribution

In [8]:
df_transactions['Payment_Method'].value_counts()

Payment_Method
Credit Card    19136
Contactless    10834
Debit Card      1683
Name: count, dtype: int64

--5 Ticket Class Distribution


In [9]:
df_transactions['Ticket_Class'].value_counts()


Ticket_Class
Standard       28595
First Class     3058
Name: count, dtype: int64

--6 Total Revenue


In [10]:
df_transactions['Price'].sum()

np.int64(741921)

--7 Revenue by Ticket Type


In [11]:
df_transactions.groupby('Ticket_Type')['Price'].sum().sort_values(ascending=False)

Ticket_Type
Advance     309274
Off-Peak    223338
Anytime     209309
Name: Price, dtype: int64

joins - merge

In [12]:
df_trans_subset = df_transactions[['Journey_ID']]
df_journeys_subset = df_journeys[['Journey_ID', 'Route_ID']]
df_routes_subset = df_routes[['Route_ID', 'Route_Name']]

In [13]:
df_sales_journeys = df_trans_subset.merge(df_journeys_subset, on='Journey_ID', how='inner')
df_sales_routes = df_sales_journeys.merge(df_routes_subset, on='Route_ID', how='inner')

--8 Top Routes by Ticket Sales


In [14]:
df_sales_routes['Route_Name'].value_counts().head(10)

Route_Name
MAN -> PAD     4628
RDG -> SWI     4209
EDB -> KGX     3922
LIV -> BHM     3873
BHM -> EUST    3471
LIV -> LEE     3002
PAD -> MAN     1097
LIV -> MAN      712
YRK -> BHM      702
BRI -> CDF      485
Name: count, dtype: int64

--9 Revenue by Ticket Class and Type


In [15]:
df_transactions.groupby(['Ticket_Class', 'Ticket_Type'])['Price'].sum().unstack(fill_value=0)

Ticket_Type,Advance,Anytime,Off-Peak
Ticket_Class,,,
First Class,66886,37841,44672
Standard,242388,171468,178666


--10 Revenue by Railcard


In [16]:
df_transactions.groupby('Railcard')['Price'].sum().sort_values(ascending=False)

Railcard
Adult       86330
Disabled    52278
Senior      29616
Name: Price, dtype: int64

--11 Revenue by Purchase Type and Payment Method


In [17]:
df_transactions.groupby(['Purchase_Type', 'Payment_Method'])['Price'].sum().unstack(fill_value=0)

Payment_Method,Contactless,Credit Card,Debit Card
Purchase_Type,,,
Online,118940,243399,20415
Station,100504,226112,32551


--12 Journey Status Distribution


In [18]:
df_journeys['Journey_status'].value_counts()

Journey_status
On Time      18019
Delayed       1062
Cancelled      790
Name: count, dtype: int64

--13 Top 10 Stations by Number of Delayed Journeys


In [19]:
delayed_journeys = df_journeys[df_journeys['Journey_status'] == 'Delayed']
delayed_journeys['departure_station'].value_counts().head(10)

departure_station
MAN     408
LIV     283
BHM     109
EUST     75
YRK      44
EDB      43
KGX      43
PAD      42
OXF      15
Name: count, dtype: int64

--14 Delay Percentage by Route


In [20]:
route_total = df_journeys.groupby('Route_ID').size()
route_total.head(10)

Route_ID
1       90
2     2412
3       45
4       15
5      361
6      161
7      118
8       14
9      120
10      15
dtype: int64

In [21]:
route_delayed = delayed_journeys.groupby('Route_ID').size()
route_delayed.head(10)

Route_ID
2      42
13     40
15     12
19     69
21     43
23     66
24    191
26     43
36    205
37     15
dtype: int64

In [22]:
delay_percentage = ((route_delayed/route_total) * 100).fillna(0).sort_values(ascending=False)
delay_percentage

Route_ID
26    100.000000
57    100.000000
41    100.000000
24     66.089965
36     56.473829
         ...    
59      0.000000
61      0.000000
62      0.000000
63      0.000000
65      0.000000
Length: 65, dtype: float64

--15 Most Common Reasons for Delay


In [23]:
df_journeys['reason_for_delay'].value_counts()

reason_for_delay
No Delay           18019
Weather              466
Staffing             437
Signal Failure       424
Technical Issue      373
Traffic              152
Name: count, dtype: int64

--16 & 17 Ticket Sales and Revenue by Lead Time


In [24]:
df_railway.info()

<class 'pandas.DataFrame'>
RangeIndex: 31653 entries, 0 to 31652
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Transaction_ID       31653 non-null  str           
 1   Date_of_Purchase     31653 non-null  datetime64[us]
 2   Time_of_Purchase     31653 non-null  object        
 3   Purchase_Type        31653 non-null  str           
 4   Payment_Method       31653 non-null  str           
 5   Railcard             10735 non-null  str           
 6   Ticket_Class         31653 non-null  str           
 7   Ticket_Type          31653 non-null  str           
 8   Price                31653 non-null  int64         
 9   Departure_Station    31653 non-null  str           
 10  Arrival_Destination  31653 non-null  str           
 11  Date_of_Journey      31653 non-null  datetime64[us]
 12  Departure_Time       31653 non-null  object        
 13  Arrival_Time         31653 non-null  objec

In [25]:
df_railway['Lead_Time_Days'] = df_railway['Date_of_Journey'] - df_railway['Date_of_Purchase']  

In [26]:
df_railway.groupby('Lead_Time_Days')['Price'].sum().sort_index()

Lead_Time_Days
0 days     432647
1 days     241899
2 days       7086
3 days       4634
4 days       4414
5 days       3872
6 days       3423
7 days       4232
8 days       4029
9 days       2744
10 days      3356
11 days      3454
12 days      2717
13 days      2449
14 days      2844
15 days      2619
16 days      2629
17 days      2669
18 days      1968
19 days      1756
20 days      1248
21 days      1161
22 days      1089
23 days      1237
24 days       500
25 days       587
26 days       323
27 days       287
28 days        48
Name: Price, dtype: int64

In [27]:
df_railway.groupby('Lead_Time_Days')['Transaction_ID'].count().sort_index()

Lead_Time_Days
0 days     14092
1 days     13744
2 days       367
3 days       284
4 days       254
5 days       221
6 days       212
7 days       253
8 days       219
9 days       182
10 days      193
11 days      190
12 days      160
13 days      149
14 days      163
15 days      148
16 days      135
17 days      119
18 days      103
19 days       91
20 days       79
21 days       74
22 days       64
23 days       52
24 days       33
25 days       36
26 days       16
27 days       19
28 days        1
Name: Transaction_ID, dtype: int64

--18 & 19 Ticket Sales and Revenue by Booking Window

In [28]:
df_window_booking = df_railway[df_railway['Purchase_Type'] == 'Station']
df_window_booking['Price'].sum()

np.int64(359167)

In [29]:
df_window_booking['Transaction_ID'].count()

np.int64(13132)

--20 Refunded Tickets and Estimated Lost Revenue by Reason for Delay

In [30]:
refunded_tickets = df_railway[df_railway['Journey_Status'] == 'Cancelled']
refunded_tickets.groupby('Reason_for_Delay')['Price'].sum().sort_index()

Reason_for_Delay
Signal Failure     12470
Staffing           11492
Technical Issue     4958
Traffic             5280
Weather            11256
Name: Price, dtype: int64

In [32]:
df_journeys['Transaction_ID'].count()


KeyError: 'Transaction_ID'

In [70]:
refunded_tickets.groupby('Reason_for_Delay')['Transaction_ID'].count().sort_index()

Reason_for_Delay
Signal Failure     519
Staffing           454
Technical Issue    235
Traffic            227
Weather            445
Name: Transaction_ID, dtype: int64

--21 Ticket Sales and Revenue by Route

--22 Ticket Sales and Revenue by Time Period



In [77]:
time_period_analysis = df_railway.groupby('Time_Period').agg(
    Tickets_Sold=('Transaction_ID', 'count'),
    Total_Revenue=('Price', 'sum')
).reset_index().sort_values('Total_Revenue', ascending=False)

In [78]:
time_period_analysis

,Time_Period,Tickets_Sold,Total_Revenue
2,Morning,11709,344642
1,Evening,8067,163727
0,Afternoon,6425,119178
3,Night,5452,114374


--23 Ticket Sales and Revenue by Day of Departure

In [82]:
day_of_departure_analysis = df_railway.groupby('Date_of_Journey').agg(
    Tickets_Sold = ('Transaction_ID', 'count'),
    Total_Revenue = ('Price', 'sum')
).reset_index().sort_values('Total_Revenue', ascending=False)

In [83]:
day_of_departure_analysis

,Date_of_Journey,Tickets_Sold,Total_Revenue
30,2024-01-31,295,9196
99,2024-04-09,302,8092
102,2024-04-12,291,7719
105,2024-04-15,298,7506
18,2024-01-19,274,7420
...,...,...,...
92,2024-04-02,153,4083
54,2024-02-24,245,3994
60,2024-03-01,73,1805
0,2024-01-01,66,1682


--24 The Most Busy Stations at Specific Times

In [89]:
df_journeys.groupby(['departure_station', 'time_period']).size().reset_index(name='Journey_Count').sort_values('Journey_Count', ascending=False)

,departure_station,time_period,Journey_Count
20,MAN,Morning,1291
28,PAD,Morning,1043
12,KGX,Morning,1000
8,EUST,Morning,999
17,LIV,Night,988
16,LIV,Morning,908
18,MAN,Afternoon,896
36,STP,Morning,728
7,EUST,Evening,713
6,EUST,Afternoon,710


--25 Monthly Performance Comparison by Route

In [90]:
df_journeys['Month_Year'] = df_journeys['date_of_journey'].dt.to_period('M')
df_monthly = df_journeys.merge(df_routes[['Route_ID', 'Route_Name']], on='Route_ID', how='inner')
monthly_perf = df_monthly.groupby(['Route_Name', 'Month_Year']).agg(
    Total_Journeys=('Journey_ID', 'count'),
    Delayed_Journeys=('Journey_status', lambda x: (x == 'Delayed').sum())
).reset_index()

In [94]:
monthly_perf.sort_values(['Delayed_Journeys', 'Month_Year'], ascending=[False, False])

,Route_Name,Month_Year,Total_Journeys,Delayed_Journeys
173,PAD -> MAN,2024-03,93,56
131,MAN -> LEE,2024-01,82,55
171,PAD -> MAN,2024-01,98,52
172,PAD -> MAN,2024-02,84,50
132,MAN -> LEE,2024-02,71,49
...,...,...,...,...
235,YRK -> EDB,2024-01,24,0
239,YRK -> EDP,2024-01,3,0
243,YRK -> LEE,2024-01,7,0
247,YRK -> LIV,2024-01,9,0
